In [ ]:
import sys
sys.path.insert(0, '/users/PGS0411/pdhanish/.conda/envs/medical_llm/lib/python3.9/site-packages')

from groq import Groq
import pickle
import os
import chromadb
from chromadb.utils import embedding_functions

GROQ_API_KEY = ""
GROQ_MODEL = "llama-3.1-8b-instant"
BASE_DIR = os.path.expanduser("~/Final project")
VECTOR_DB_DIR = os.path.join(BASE_DIR, "vector_db")
BM25_DIR = os.path.join(BASE_DIR, "bm25_index")

client = Groq(api_key=GROQ_API_KEY)
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
ef = embedding_functions.DefaultEmbeddingFunction()
collection = chroma_client.get_or_create_collection("medical_qa", embedding_function=ef)

with open(os.path.join(BM25_DIR, "bm25_index.pkl"), "rb") as f:
    bm25, texts = pickle.load(f)

print("✓ Setup complete")

✓ Setup complete


In [7]:
from rank_bm25 import BM25Okapi
import pandas as pd
import random
import os

# Define paths
BASE_DIR = os.path.expanduser("~/Final project")
DATA_DIR = os.path.join(BASE_DIR, "data")

# Load test cases dynamically from dataset — no hardcoding
df_eval = pd.read_csv(os.path.join(DATA_DIR, "medical_qa.csv"))
sampled = df_eval.sample(n=5, random_state=42)
test_cases = [
    {
        "question": row["question"],
        "expected": row["answer"][:50]
    }
    for _, row in sampled.iterrows()
]
print(f"✓ Loaded {len(test_cases)} test cases dynamically from dataset")
# Load test cases dynamically from dataset — no hardcoding
df_eval = pd.read_csv(os.path.join(DATA_DIR, "medical_qa.csv"))
sampled = df_eval.sample(n=5, random_state=42)
test_cases = [
    {
        "question": row["question"],
        "expected": row["answer"][:50]
    }
    for _, row in sampled.iterrows()
]
print(f"✓ Loaded {len(test_cases)} test cases dynamically from dataset")

def hybrid_search_eval(query, n_results=5):
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_idx = sorted(range(len(bm25_scores)),
                          key=lambda i: bm25_scores[i], reverse=True)[:n_results]
    bm25_results = [texts[i] for i in top_bm25_idx]
    semantic_results = collection.query(query_texts=[query], n_results=n_results)
    semantic_docs = semantic_results['documents'][0]
    distance = semantic_results['distances'][0][0]
    confidence = round(max(0, (2 - distance) / 2 * 100), 1)
    combined = list(dict.fromkeys(bm25_results + semantic_docs))[:n_results]
    return combined, confidence

def generate_answer(query, context):
    prompt = f"""You are MediQ, an AI medical assistant.
Answer in exactly 2 sentences. Be direct and concise.
Medical Context: {context}
Question: {query}
Answer:"""
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )
    return response.choices[0].message.content

# Run evaluation
results = []
for case in test_cases:
    docs, confidence = hybrid_search_eval(case["question"])
    context = "\n".join(docs[:3])
    answer = generate_answer(case["question"], context)

    expected_lower = case["expected"].lower()
    answer_lower = answer.lower()
    relevant = any(word in answer_lower for word in expected_lower.split()[:3])

    results.append({
        "Question": case["question"][:50] + "...",
        "Confidence": f"{confidence}%",
        "Answer Preview": answer[:80] + "...",
        "Relevant": "✓" if relevant else "✗"
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print(f"\nRelevance Score: {sum(1 for r in results if r['Relevant'] == '✓')}/{len(results)}")

✓ Loaded 5 test cases dynamically from dataset
✓ Loaded 5 test cases dynamically from dataset


2026-05-06 23:43:15.359840162 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 907789, index: 6, mask: {18, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-06 23:43:15.368446413 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 907790, index: 7, mask: {14, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-06 23:43:15.373427318 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 907791, index: 8, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-06 23:43:15.378427257 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 907792, index: 9, mask: {20, }, error code: 22 error msg: Invalid argument. Specify the n

                                             Question Confidence                                                                      Answer Preview Relevant
  Are osteoclasts specialized in bone degradation?...      88.5% Osteoclasts are specialized in bone degradation, functioning as the normal bone-...        ✓
Which mushroom is poisonous, Amanita phalloides or...      93.0% Amanita phalloides is a poisonous mushroom, while Agaricus bisporus is safe to e...        ✓
                      Is AZD9668 a VEGF mRNA drug?...      87.0% No, AZD9668 is a reversible and selective inhibitor of neutrophil elastase. It d...        ✓
What is the Genomic Regions Enrichment of Annotati...      98.2% The Genomic Regions Enrichment of Annotations Tool (GREAT) is a bioinformatics t...        ✓
                       Who should wear dosimeters?...      73.4% Nuclear medicine technologists and medical staff who frequently work with medica...        ✓

Relevance Score: 5/5
